# IE 306 - Assignment 2: Coordinated Traffic Corridor Simulation

This notebook runs the full assignment experiment using the implementation in `assignment.py`.

Scenarios:
1. Uncoordinated, no emergencies (`offset=0`, emergencies OFF)
2. Coordinated, no emergencies (`offset=20`, emergencies OFF)
3. Coordinated, with emergencies (`offset=20`, emergencies ON)

Each scenario is run with 20 independent replications, warm-up 900s, run length 7200s, and 95% confidence intervals.


## 1) Setup
Make sure this notebook is in the same folder as `assignment.py`.


In [ ]:
import math
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist

import assignment
importlib.reload(assignment)

print('Loaded module from:', Path(assignment.__file__).resolve())
print('Base seed:', assignment.BASE_SEED)
print('Parameters:', assignment.PARAMS)


## 2) Scenario Runner and CI Summary

In [ ]:
def run_scenario_df(offset, emergencies_enabled, n_reps=assignment.PARAMS['n_reps'], base_seed=assignment.BASE_SEED):
    rows = []
    for rep in range(n_reps):
        seed = base_seed + rep * 100
        kpis = assignment.run_replication(seed=seed, offset=offset, enable_emergency=emergencies_enabled)
        rows.append(kpis)
    return pd.DataFrame(rows)


def summarize_with_ci(df):
    out = {}
    for col in df.columns:
        vals = df[col].to_numpy(dtype=float)
        n = len(vals)
        mean = float(vals.mean())
        if n > 1:
            se = float(vals.std(ddof=1) / math.sqrt(n))
            half = float(t_dist.ppf(0.975, df=n-1) * se)
        else:
            half = 0.0
        out[col] = {
            'mean': mean,
            'ci_half': half,
            'ci_low': mean - half,
            'ci_high': mean + half,
        }
    return pd.DataFrame(out).T


## 3) Run All Three Scenarios

In [ ]:
N_REPS = assignment.PARAMS['n_reps']
BASE_SEED = assignment.BASE_SEED
PHI = assignment.PARAMS['phi']

df_s1 = run_scenario_df(offset=0, emergencies_enabled=False, n_reps=N_REPS, base_seed=BASE_SEED)
df_s2 = run_scenario_df(offset=PHI, emergencies_enabled=False, n_reps=N_REPS, base_seed=BASE_SEED)
df_s3 = run_scenario_df(offset=PHI, emergencies_enabled=True, n_reps=N_REPS, base_seed=BASE_SEED)

summary_s1 = summarize_with_ci(df_s1)
summary_s2 = summarize_with_ci(df_s2)
summary_s3 = summarize_with_ci(df_s3)

print('Completed scenario runs.')
print('Rows per scenario:', len(df_s1), len(df_s2), len(df_s3))


## 4) Comparative KPI Table (Mean ± 95% CI)
The table below focuses on assignment KPIs.

In [ ]:
KPI_ORDER = [
    'avg_delay_through', 'avg_delay_WE',
    'throughput_per_hour', 'avg_link_occupancy', 'blocking_frequency',
    'avg_queue_ns_A', 'max_queue_ns_A',
    'avg_queue_ns_B', 'max_queue_ns_B',
    'avg_queue_we_A', 'max_queue_we_A',
    'avg_queue_we_B', 'max_queue_we_B',
    'preemptions_total', 'avg_recovery_time',
]


def format_scenario(summary_df, label):
    vals = {}
    for k in KPI_ORDER:
        m = summary_df.loc[k, 'mean']
        h = summary_df.loc[k, 'ci_half']
        vals[k] = f"{m:,.3f} +/- {h:,.3f}"
    return pd.Series(vals, name=label)

comparative = pd.concat([
    format_scenario(summary_s1, 'S1 Uncoord'),
    format_scenario(summary_s2, 'S2 Coord'),
    format_scenario(summary_s3, 'S3 Coord+Emerg'),
], axis=1)

comparative


## 5) Key Plots

In [ ]:
plot_kpis = [
    ('avg_delay_through', 'Avg Delay S->N Through (s)'),
    ('avg_delay_WE', 'Avg Delay W/E (s)'),
    ('throughput_per_hour', 'Throughput at North Exit (veh/h)'),
    ('avg_link_occupancy', 'Average Link Occupancy (veh)'),
    ('preemptions_total', 'Preemptions (count)'),
    ('avg_recovery_time', 'Recovery Time (s)'),
]

labels = ['S1', 'S2', 'S3']
means = {
    'S1': summary_s1['mean'].to_dict(),
    'S2': summary_s2['mean'].to_dict(),
    'S3': summary_s3['mean'].to_dict(),
}
ci = {
    'S1': summary_s1['ci_half'].to_dict(),
    'S2': summary_s2['ci_half'].to_dict(),
    'S3': summary_s3['ci_half'].to_dict(),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for ax, (kpi, title) in zip(axes, plot_kpis):
    y = [means[s].get(kpi, 0.0) for s in labels]
    e = [ci[s].get(kpi, 0.0) for s in labels]
    ax.bar(labels, y, yerr=e, capsize=4)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 6) Verification / Validation Checks (Quick)
These are quick sanity checks for report section 2.e.

In [ ]:
checks = {}
checks['phi_equals_L_over_vf'] = abs(assignment.PARAMS['phi'] - assignment.PARAMS['L'] / assignment.PARAMS['vf']) < 1e-12
checks['warmup_is_900'] = assignment.PARAMS['warmup'] == 900
checks['sim_time_is_7200'] = assignment.PARAMS['sim_time'] == 7200
checks['scenario3_has_preemptions'] = summary_s3.loc['preemptions_total', 'mean'] > 0
checks['blocking_frequency_nonnegative'] = (
    summary_s1.loc['blocking_frequency', 'mean'] >= 0 and
    summary_s2.loc['blocking_frequency', 'mean'] >= 0 and
    summary_s3.loc['blocking_frequency', 'mean'] >= 0
)

pd.Series(checks, name='pass')


## 7) Optional: Save Outputs
Uncomment if you want csv exports.

```python
# comparative.to_csv('kpi_comparative_table.csv')
# df_s1.to_csv('scenario1_replications.csv', index=False)
# df_s2.to_csv('scenario2_replications.csv', index=False)
# df_s3.to_csv('scenario3_replications.csv', index=False)
```
